In [ ]:
from IPython.display import display, HTML

display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;600;700&display=swap');

body, .jp-Notebook, .vh-container {
    font-family: 'Poppins', sans-serif !important;
    background: linear-gradient(135deg, #eef2f7 0%, #dbe6f0 100%);
}

.landmark-banner {
    background: linear-gradient(120deg, #2c3e50 0%, #4b6cb7 100%);
    color: white;
    padding: 32px 40px;
    border-radius: 18px;
    margin-bottom: 24px;
    box-shadow: 0 10px 30px rgba(44, 62, 80, 0.25);
    text-align: center;
}
.landmark-banner h1 {
    margin: 0 0 8px 0;
    font-size: 32px;
    font-weight: 700;
    letter-spacing: 0.5px;
}
.landmark-banner p {
    margin: 0;
    font-size: 15px;
    opacity: 0.9;
    font-weight: 400;
}

.landmark-card {
    background: white;
    border-radius: 16px;
    padding: 28px 32px;
    margin-bottom: 20px;
    box-shadow: 0 6px 18px rgba(0,0,0,0.06);
}

.landmark-section-title {
    font-size: 18px;
    font-weight: 600;
    color: #2c3e50;
    margin-bottom: 14px;
    border-left: 5px solid #4b6cb7;
    padding-left: 10px;
}

.pred-row { margin: 10px 0; }
.pred-row .pred-label {
    display: flex;
    justify-content: space-between;
    font-size: 14px;
    color: #2c3e50;
    margin-bottom: 4px;
}
.pred-row .pred-name { font-weight: 600; }
.pred-row .pred-bar-bg {
    background: #e7ecf3;
    border-radius: 10px;
    height: 16px;
    overflow: hidden;
}
.pred-row .pred-bar-fill {
    height: 100%;
    border-radius: 10px;
    background: linear-gradient(90deg, #4facfe 0%, #00c2ba 100%);
    transition: width 0.6s ease;
}
.pred-row.top1 .pred-bar-fill {
    background: linear-gradient(90deg, #ff8a00 0%, #e52e71 100%);
}

.landmark-footer {
    text-align: center;
    color: #7a8699;
    font-size: 12.5px;
    margin-top: 10px;
    padding-bottom: 20px;
}

.widget-button {
    border-radius: 10px !important;
}
</style>

<div class="landmark-banner">
    <h1>🏛️ Landmark Snap</h1>
    <p>Upload a photo of a landmark and let the model guess where it is</p>
</div>
"""))

In [ ]:
import io
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as T

# The exported TorchScript model is fully self-contained: it already
# bundles the preprocessing transforms, the trained weights, and the
# list of class names (see src/predictor.py in the training project).
MODEL_PATH = "checkpoints/transfer_exported.pt"
learn_inf = torch.jit.load(MODEL_PATH, map_location="cpu")
learn_inf.eval()


def format_landmark_name(name: str) -> str:
    """Turn '09.Golden_Gate_Bridge' into 'Golden Gate Bridge'."""
    if "." in name:
        name = name.split(".", 1)[1]
    return name.replace("_", " ")


def render_predictions_html(softmax: np.ndarray, class_names, top_k: int = 5) -> str:
    idxs = np.argsort(softmax)[::-1][:top_k]
    rows = []
    for rank, idx in enumerate(idxs):
        prob = float(softmax[idx]) * 100
        name = format_landmark_name(class_names[idx])
        top_class = " top1" if rank == 0 else ""
        rows.append(f"""
        <div class="pred-row{top_class}">
            <div class="pred-label">
                <span class="pred-name">{rank + 1}. {name}</span>
                <span>{prob:.1f}%</span>
            </div>
            <div class="pred-bar-bg">
                <div class="pred-bar-fill" style="width:{prob:.1f}%;"></div>
            </div>
        </div>
        """)
    return "".join(rows)


def classify_image(img: Image.Image):
    """Run the model on a PIL image and return (html_predictions, top_label)."""
    img = img.convert("RGB")
    timg = T.ToTensor()(img).unsqueeze_(0)
    with torch.no_grad():
        softmax = learn_inf(timg).data.cpu().numpy().squeeze()
    class_names = learn_inf.class_names
    html = render_predictions_html(softmax, class_names, top_k=5)
    top_label = format_landmark_name(class_names[int(np.argmax(softmax))])
    return html, top_label

In [ ]:
from ipywidgets import (
    VBox, HBox, Button, FileUpload, Output, HTML as HTMLWidget, Text, Layout
)

display(HTML('<div class="landmark-card"><div class="landmark-section-title">📷 Upload a photo</div></div>'))

btn_upload = FileUpload(accept="image/*", multiple=False)
btn_classify = Button(description="Classify", button_style="success", icon="search")
out_image = Output()
out_predictions = Output()
out_status = Output()


def _extract_upload_bytes(uploader: FileUpload):
    """
    Handle both ipywidgets 7.x (uploader.data is a list of raw bytes) and
    ipywidgets 8.x (uploader.data was removed; use uploader.value instead,
    which is a tuple of dicts with a 'content' key) FileUpload APIs.
    Returns None if no file (or an empty file) was received.
    """
    if hasattr(uploader, "data") and uploader.data:
        content = uploader.data[-1]
    else:
        uploaded = uploader.value
        if not uploaded:
            return None
        if isinstance(uploaded, dict):
            content = list(uploaded.values())[-1]["content"]
        else:
            content = uploaded[-1]["content"]
    content = bytes(content)
    return content if len(content) > 0 else None


def _show_status(message, kind="info"):
    colors = {"info": "#4b6cb7", "error": "#e74c3c", "success": "#27ae60"}
    out_status.clear_output()
    with out_status:
        display(HTML(
            f'<div style="color:{colors.get(kind, "#4b6cb7")}; '
            f'font-size:13px; margin-top:6px;">{message}</div>'
        ))


def _run_classification(img):
    out_image.clear_output()
    with out_image:
        ratio = img.size[0] / img.size[1]
        thumb = img.copy()
        thumb.thumbnail([ratio * 260, 260])
        display(thumb)

    html_predictions, top_label = classify_image(img)
    out_predictions.clear_output()
    with out_predictions:
        display(HTML(
            '<div class="landmark-section-title">🏆 Best guess: '
            f'{top_label}</div>{html_predictions}'
        ))
    _show_status("Done!", kind="success")


def on_click_classify(change):
    _show_status("Classifying...", kind="info")
    content = _extract_upload_bytes(btn_upload)

    if content is None:
        _show_status(
            "No image data was received from the upload widget. This can happen "
            "in some notebook front-ends -- please use the 'Classify from a path' "
            "option below instead.",
            kind="error",
        )
        return

    try:
        img = Image.open(io.BytesIO(content))
        img.load()
    except Exception as e:
        _show_status(f"Could not read the uploaded file as an image: {e}", kind="error")
        return

    _run_classification(img)


btn_classify.on_click(on_click_classify)

upload_row = HBox([btn_upload, btn_classify])
display(VBox([upload_row, out_status, out_image, out_predictions]))

In [ ]:
display(HTML(
    '<div class="landmark-card">'
    '<div class="landmark-section-title">🔗 Or classify from a URL</div>'
    '<p style="font-size:13px;color:#7a8699;margin-top:-6px;">'
    "If the upload button above doesn't work in your browser, paste a direct "
    "image URL (ending in .jpg/.png) below instead."
    "</p></div>"
))

import urllib.request

url_input = Text(
    placeholder="https://example.com/photo.jpg",
    layout=Layout(width="420px"),
)
btn_classify_url = Button(description="Classify URL", button_style="info", icon="link")
out_url_status = Output()


def on_click_classify_url(change):
    out_url_status.clear_output()
    url = url_input.value.strip()
    if not url:
        with out_url_status:
            display(HTML('<div style="color:#e74c3c;font-size:13px;">Please paste an image URL first.</div>'))
        return
    try:
        with out_url_status:
            display(HTML('<div style="color:#4b6cb7;font-size:13px;">Downloading and classifying...</div>'))
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=15) as response:
            data = response.read()
        img = Image.open(io.BytesIO(data))
        img.load()
    except Exception as e:
        out_url_status.clear_output()
        with out_url_status:
            display(HTML(f'<div style="color:#e74c3c;font-size:13px;">Could not load that image: {e}</div>'))
        return

    out_url_status.clear_output()
    _run_classification(img)


btn_classify_url.on_click(on_click_classify_url)
display(VBox([HBox([url_input, btn_classify_url]), out_url_status]))

In [ ]:
display(HTML(
    '<div class="landmark-footer">Built with a CNN + transfer learning (ResNet18) '
    "on a 50-landmark subset of the Google Landmarks dataset &middot; "
    'Served with <a href="https://voila.readthedocs.io" target="_blank">Voil\u00e0</a></div>'
))